# 01 — Build `tomato_ripeness_v1` (Kaggle)

**Mục tiêu cuối cùng của notebook này:** tạo ra **một bộ dataset hoàn chỉnh** `tomato_ripeness_v1` (ảnh + label YOLO 3 class + toàn bộ manifest) sẵn sàng **Save Version thành Kaggle Dataset**, để notebook train baseline sau này chỉ cần Add Input là dùng được ngay — không cần tải/xử lý lại.

## Trước khi chạy
1. **Add Input** → attach output đã Save Version của `00_data_acquisition_ripeness.ipynb` (chứa `ai/datasets/raw/{laboro_tomato,agrob_tomato,tomato_plantfactory,openfield_bd}`).
2. Panel phải → **Internet: ON** (cần để `pip install imagehash`).
3. Accelerator: **None** — notebook này không train.

## Notebook này làm gì
1. Tự nhận diện thư mục input + khám phá cấu trúc thật của từng nguồn (không đoán mò đường dẫn).
2. Liệt kê **tên class GỐC** thật sự có trong dữ liệu — đối chiếu với class mapping trước khi tin tưởng (đúng nguyên tắc "không remap chỉ dựa vào tên").
3. Convert 4 nguồn (Laboro Tomato: COCO, AgRobTomato: Pascal VOC, TomatoPlantfactory: YOLO, OpenField-BD: YOLO) sang **YOLO 3-class thống nhất**: `fruit_green_unripe` / `fruit_turning` / `fruit_ripe`.
4. Phát hiện ảnh trùng/near-duplicate (SHA-256 + pHash) để nhóm ảnh trùng vào cùng 1 split.
5. Chia `train/val/test` (75/15/10) theo nhóm trùng lặp + theo nguồn — chống data leakage. **OpenField-BD bị tách riêng** thành `test_outdomain_openfield/` (license chưa xác nhận → không train).
6. Sinh đầy đủ manifest: `data.yaml`, `class_mapping.yaml`, `class_distribution.csv`, `source_distribution.csv`, `split_manifest.csv`, `duplicate_report.csv`, `review_required.csv`, `rejected_images.csv`, `licenses.md`, `dataset_report.md`.
7. Vẽ mẫu ảnh + bounding box để xác nhận trực quan (đặc biệt là 2 nguồn dùng class-id dạng số, nơi phải **giả định** thứ tự id).

In [1]:
import subprocess
subprocess.run(["pip", "install", "-q", "imagehash"], check=False)
subprocess.run(["pip", "install", "-q", "pyyaml"], check=False)

CompletedProcess(args=['pip', 'install', '-q', 'pyyaml'], returncode=0)

In [2]:
import hashlib
import json
import random
import re
import shutil
import time
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path

import imagehash
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image

random.seed(42)
np.random.seed(42)

WORKING = Path("/kaggle/working/ai/datasets")
PROCESSED = WORKING / "processed" / "tomato_ripeness_v1"
INTERIM = WORKING / "interim"
MANIFESTS = WORKING / "manifests"

for d in [PROCESSED, INTERIM, MANIFESTS]:
    d.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = ["fruit_green_unripe", "fruit_turning", "fruit_ripe"]
CLASS_NAME_TO_ID = {name: i for i, name in enumerate(TARGET_CLASSES)}
REVIEW_REQUIRED = "REVIEW_REQUIRED"

print("PROCESSED:", PROCESSED)

PROCESSED: /kaggle/working/ai/datasets/processed/tomato_ripeness_v1


In [3]:
candidates = sorted(Path("/kaggle/input").glob("*/ai/datasets/raw"))
print("Các input khả dĩ tìm thấy:")
for c in candidates:
    print(" -", c)

# Đặt thủ công nếu auto-detect chọn sai, ví dụ:
# INPUT_OVERRIDE = Path("/kaggle/input/00-data-acquisition-ripeness/ai/datasets/raw")
INPUT_OVERRIDE = None

if INPUT_OVERRIDE is not None:
    RAW_INPUT = INPUT_OVERRIDE
elif candidates:
    RAW_INPUT = candidates[0]
else:
    raise FileNotFoundError(
        "Không tìm thấy raw data. Hãy Add Input output đã Save Version của "
        "00_data_acquisition_ripeness.ipynb trước khi chạy notebook này."
    )

print("\nDùng RAW_INPUT =", RAW_INPUT)

SOURCE_DIRS = {
    "laboro_tomato": RAW_INPUT / "laboro_tomato",
    "agrob_tomato": RAW_INPUT / "agrob_tomato",
    "tomato_plantfactory": RAW_INPUT / "tomato_plantfactory",
    "openfield_bd": RAW_INPUT / "openfield_bd",
}
for name, p in SOURCE_DIRS.items():
    print(f"  [{'OK' if p.exists() else 'THIẾU'}] {name}: {p}")

Các input khả dĩ tìm thấy:


FileNotFoundError: Không tìm thấy raw data. Hãy Add Input output đã Save Version của 00_data_acquisition_ripeness.ipynb trước khi chạy notebook này.

### Bước 1 — Khám phá cấu trúc thư mục thật (không đoán mò)
In cây thư mục + đếm file theo đuôi + tìm file định nghĩa class (`classes.txt`/`*.yaml`/`*.names`) cho 2 nguồn dùng class-id dạng số.

In [ ]:
def ext_counts(root):
    counts = defaultdict(int)
    for p in Path(root).rglob("*"):
        if p.is_file():
            counts[p.suffix.lower()] += 1
    return dict(sorted(counts.items(), key=lambda kv: -kv[1]))


def describe_structure(root, max_depth=3, samples=3):
    root = Path(root)
    if not root.exists():
        print(f"  (không tồn tại: {root})")
        return
    dirs = sorted({p for p in root.rglob("*") if p.is_dir() and len(p.relative_to(root).parts) <= max_depth})
    for d in [root] + dirs:
        depth = len(d.relative_to(root).parts)
        if depth > max_depth:
            continue
        files = sorted(f.name for f in d.iterdir() if f.is_file())
        label = d.relative_to(root) if d != root else "."
        indent = "  " * depth
        print(f"{indent}{label}/  ({len(files)} file)")
        for f in files[:samples]:
            print(f"{indent}  - {f}")
        if len(files) > samples:
            print(f"{indent}  ... và {len(files) - samples} file khác")


for name, path in SOURCE_DIRS.items():
    print(f"\n===== {name} =====")
    print("Đếm file theo đuôi:", ext_counts(path))
    describe_structure(path)

print("\n===== Tìm file định nghĩa class cho nguồn dùng class-id dạng số =====")
for name in ["tomato_plantfactory", "openfield_bd"]:
    root = SOURCE_DIRS[name]
    found = list(root.rglob("classes.txt")) + list(root.rglob("*.yaml")) + list(root.rglob("*.yml")) + list(root.rglob("*.names"))
    print(f"\n-- {name} --")
    if not found:
        print("  Không tìm thấy -> dùng thứ tự GIẢ ĐỊNH theo tài liệu dự án, sẽ xác nhận lại bằng ảnh mẫu ở Bước 8.")
    for f in found:
        print(" ", f)
        try:
            print("   nội dung:", f.read_text(encoding="utf-8", errors="ignore")[:300])
        except Exception as e:
            print("   (không đọc được:", e, ")")

### Bước 2 — Liệt kê tên class GỐC thật sự có trong dữ liệu
So sánh danh sách in ra dưới đây với `name_map` ở Bước 3. Nếu khác nhau (chính tả, hoa/thường...), phải sửa `name_map` trước khi chạy tiếp — **không remap chỉ dựa vào phỏng đoán**.

In [ ]:
ASSUMED_ID_TO_NAME = {
    "tomato_plantfactory": {0: "green", 1: "red"},
    "openfield_bd": {0: "green", 1: "half_ripe", 2: "fully_ripe"},
}
SOURCE_FORMAT = {
    "laboro_tomato": "coco",
    "agrob_tomato": "voc",
    "tomato_plantfactory": "yolo",
    "openfield_bd": "yolo",
}


def collect_raw_class_names(name):
    root = SOURCE_DIRS[name]
    fmt = SOURCE_FORMAT[name]
    names = set()
    if fmt == "coco":
        for jf in root.rglob("*.json"):
            with open(jf, encoding="utf-8") as f:
                data = json.load(f)
            for c in data.get("categories", []):
                names.add(c["name"])
    elif fmt == "voc":
        for xf in root.rglob("*.xml"):
            try:
                for obj in ET.parse(xf).getroot().findall("object"):
                    names.add(obj.find("name").text)
            except Exception:
                pass
    elif fmt == "yolo":
        ids = set()
        for tf in root.rglob("*.txt"):
            for line in tf.read_text().splitlines():
                parts = line.split()
                if parts:
                    ids.add(int(parts[0]))
        id_to_name = ASSUMED_ID_TO_NAME.get(name, {})
        names = {f"id={i} (giả định: {id_to_name.get(i, '???')})" for i in sorted(ids)}
    return names


for name in SOURCE_DIRS:
    print(f"{name}: {sorted(collect_raw_class_names(name))}")

### Bước 3 — Class mapping chuẩn
Theo `TONG_HOP_DATASET_DO_CHIN_CA_CHUA_PUBLIC.docx` + `HUONG_DAN_XAY_DUNG_DATASET_SMART_GREENHOUSE_AI.md`. `"reddish"` của AgRobTomato để `REVIEW_REQUIRED` (không tự remap) — box này sẽ bị loại khỏi label và ghi vào `review_required.csv` để xem lại thủ công.

In [ ]:
CLASS_MAPPING = {
    "laboro_tomato": {
        "format": "coco",
        "name_map": {
            "b_green": "fruit_green_unripe", "l_green": "fruit_green_unripe",
            "b_half_ripened": "fruit_turning", "l_half_ripened": "fruit_turning",
            "b_fully_ripened": "fruit_ripe", "l_fully_ripened": "fruit_ripe",
        },
    },
    "agrob_tomato": {
        "format": "voc",
        "name_map": {
            "unriped": "fruit_green_unripe",
            "breaking_stage": "fruit_turning",
            "riped": "fruit_ripe",
            "reddish": REVIEW_REQUIRED,
        },
    },
    "tomato_plantfactory": {
        "format": "yolo",
        "id_to_name": ASSUMED_ID_TO_NAME["tomato_plantfactory"],
        "name_map": {"green": "fruit_green_unripe", "red": "fruit_ripe"},
    },
    "openfield_bd": {
        "format": "yolo",
        "id_to_name": ASSUMED_ID_TO_NAME["openfield_bd"],
        "name_map": {
            "green": "fruit_green_unripe",
            "half_ripe": "fruit_turning",
            "fully_ripe": "fruit_ripe",
        },
        "outdomain_test_only": True,
    },
}

mapping_out = {"target_classes": TARGET_CLASSES, **CLASS_MAPPING}
with open(MANIFESTS / "class_mapping.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(mapping_out, f, allow_unicode=True, sort_keys=False)

print((MANIFESTS / "class_mapping.yaml").read_text(encoding="utf-8"))

### Bước 4 — Parser theo định dạng (COCO / Pascal VOC / YOLO)
Tự nhận diện file theo đuôi/nội dung, không hardcode đường dẫn cụ thể — chạy được kể cả nếu cấu trúc thư mục con hơi khác giữa các nguồn.

In [ ]:
def normalize_name(s):
    s = s.strip().lower()
    return re.sub(r"[\s_-]+", "_", s)


def find_images(root):
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    return {p.stem: p for p in Path(root).rglob("*") if p.suffix.lower() in exts}


def parse_coco(json_path):
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    cats = {c["id"]: c["name"] for c in data["categories"]}
    images = {im["id"]: im for im in data["images"]}
    per_image_boxes = defaultdict(list)
    for ann in data["annotations"]:
        im = images.get(ann["image_id"])
        if im is None:
            continue
        x, y, w, h = ann["bbox"]
        cname = cats.get(ann["category_id"], "UNKNOWN")
        stem = Path(im["file_name"]).stem
        per_image_boxes[stem].append((cname, x, y, x + w, y + h))
    result = {}
    for im in images.values():
        stem = Path(im["file_name"]).stem
        result[stem] = (im["width"], im["height"], per_image_boxes.get(stem, []))
    return result


def parse_voc(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    w, h = int(size.find("width").text), int(size.find("height").text)
    boxes = []
    for obj in root.findall("object"):
        cname = obj.find("name").text
        bnd = obj.find("bndbox")
        xmin, ymin = float(bnd.find("xmin").text), float(bnd.find("ymin").text)
        xmax, ymax = float(bnd.find("xmax").text), float(bnd.find("ymax").text)
        boxes.append((cname, xmin, ymin, xmax, ymax))
    return w, h, boxes


def parse_yolo(txt_path, img_w, img_h, id_to_name):
    boxes = []
    for line in Path(txt_path).read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        cname = id_to_name.get(cid, "UNKNOWN")
        xmin, ymin = (xc - bw / 2) * img_w, (yc - bh / 2) * img_h
        xmax, ymax = (xc + bw / 2) * img_w, (yc + bh / 2) * img_h
        boxes.append((cname, xmin, ymin, xmax, ymax))
    return boxes

In [ ]:
def load_source_annotations(name, cfg):
    """stem -> (img_path, img_w, img_h, boxes[(class_name, xmin, ymin, xmax, ymax)])"""
    root = SOURCE_DIRS[name]
    images = find_images(root)
    fmt = cfg["format"]
    result = {}

    if fmt == "coco":
        coco_data = {}
        for jf in root.rglob("*.json"):
            coco_data.update(parse_coco(jf))
        for stem, (w, h, boxes) in coco_data.items():
            if stem in images:
                result[stem] = (images[stem], w, h, boxes)

    elif fmt == "voc":
        xml_by_stem = {p.stem: p for p in root.rglob("*.xml")}
        for stem, img_path in images.items():
            xml_path = xml_by_stem.get(stem)
            if xml_path is None:
                continue
            w, h, boxes = parse_voc(xml_path)
            result[stem] = (img_path, w, h, boxes)

    elif fmt == "yolo":
        txt_by_stem = {p.stem: p for p in root.rglob("*.txt")}
        id_to_name = cfg["id_to_name"]
        for stem, img_path in images.items():
            txt_path = txt_by_stem.get(stem)
            if txt_path is None:
                continue
            with Image.open(img_path) as im:
                w, h = im.size
            boxes = parse_yolo(txt_path, w, h, id_to_name)
            result[stem] = (img_path, w, h, boxes)

    else:
        raise ValueError(f"Format chưa hỗ trợ: {fmt}")

    return result


def remap_boxes(boxes, img_w, img_h, name_map):
    lines, n_review, n_drop = [], 0, 0
    for cname, xmin, ymin, xmax, ymax in boxes:
        target = name_map.get(normalize_name(cname))
        if target is None:
            n_drop += 1
            continue
        if target == REVIEW_REQUIRED:
            n_review += 1
            continue
        xmin, xmax = max(0, xmin), min(img_w, xmax)
        ymin, ymax = max(0, ymin), min(img_h, ymax)
        if xmax <= xmin or ymax <= ymin:
            n_drop += 1
            continue
        xc, yc = (xmin + xmax) / 2 / img_w, (ymin + ymax) / 2 / img_h
        bw, bh = (xmax - xmin) / img_w, (ymax - ymin) / img_h
        lines.append(f"{CLASS_NAME_TO_ID[target]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    return lines, n_review, n_drop


def convert_source(name, cfg, out_dir):
    annotations = load_source_annotations(name, cfg)
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)

    stats = {"images_total": len(annotations), "images_kept": 0, "boxes_kept": 0,
             "boxes_dropped_unmapped": 0, "boxes_review": 0}
    review_rows, rejected_rows = [], []

    for stem, (img_path, w, h, boxes) in annotations.items():
        lines, n_review, n_drop = remap_boxes(boxes, w, h, cfg["name_map"])
        stats["boxes_review"] += n_review
        stats["boxes_dropped_unmapped"] += n_drop
        if n_review:
            review_rows.append({"source": name, "image": img_path.name, "n_review_boxes": n_review})
        if not lines:
            rejected_rows.append({"source": name, "image": img_path.name, "reason": "khong_con_box_hop_le_sau_remap"})
            continue
        # Prefix bằng source_id để không đụng tên khi gộp nhiều nguồn vào cùng 1 thư mục split
        dest_img = out_dir / "images" / f"{name}__{img_path.name}"
        shutil.copy2(img_path, dest_img)
        (out_dir / "labels" / f"{name}__{img_path.stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        stats["images_kept"] += 1
        stats["boxes_kept"] += len(lines)

    return stats, review_rows, rejected_rows

In [ ]:
all_stats = {}
all_review_rows = []
all_rejected_rows = []

for name, cfg in CLASS_MAPPING.items():
    out_dir = INTERIM / f"{name}_normalized"
    print(f"\n== Convert {name} ==")
    stats, review_rows, rejected_rows = convert_source(name, cfg, out_dir)
    all_stats[name] = stats
    all_review_rows.extend(review_rows)
    all_rejected_rows.extend(rejected_rows)
    print(stats)

stats_df = pd.DataFrame(all_stats).T
print("\n", stats_df)

pd.DataFrame(all_review_rows).to_csv(MANIFESTS / "review_required.csv", index=False)
pd.DataFrame(all_rejected_rows).to_csv(MANIFESTS / "rejected_images.csv", index=False)
print(f"\nreview_required.csv: {len(all_review_rows)} dòng | rejected_images.csv: {len(all_rejected_rows)} dòng")

### Bước 5 — Phát hiện ảnh trùng / near-duplicate
SHA-256 cho trùng chính xác + pHash (Hamming distance ≤ 6) cho near-duplicate, dùng union-find để gom nhóm. Toàn bộ ảnh cùng nhóm phải nằm cùng 1 split ở Bước 6.

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()


all_images = []
for name in CLASS_MAPPING:
    img_dir = INTERIM / f"{name}_normalized" / "images"
    if img_dir.exists():
        all_images.extend((name, p) for p in sorted(img_dir.glob("*")))

print(f"Tổng số ảnh sau convert (tất cả nguồn, kể cả openfield_bd): {len(all_images)}")

records = []
for source, path in all_images:
    sha = sha256_file(path)
    try:
        phash = str(imagehash.phash(Image.open(path)))
    except Exception:
        phash = None
    records.append({"source": source, "path": str(path), "filename": path.name, "sha256": sha, "phash": phash})

img_df = pd.DataFrame(records)
print("Đã hash xong", len(img_df), "ảnh.")

In [ ]:
n = len(img_df)
parent = list(range(n))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def union(i, j):
    ri, rj = find(i), find(j)
    if ri != rj:
        parent[ri] = rj


sha_groups = defaultdict(list)
for i, sha in enumerate(img_df["sha256"]):
    sha_groups[sha].append(i)
for idxs in sha_groups.values():
    for j in idxs[1:]:
        union(idxs[0], j)

phash_ints = [int(p, 16) if p else None for p in img_df["phash"]]
THRESH = 6
t0 = time.time()
for i in range(n):
    if phash_ints[i] is None:
        continue
    for j in range(i + 1, n):
        if phash_ints[j] is None:
            continue
        if bin(phash_ints[i] ^ phash_ints[j]).count("1") <= THRESH:
            union(i, j)
print(f"So khớp pHash {n}x{n}: {time.time() - t0:.1f}s")

img_df["duplicate_group_id"] = [find(i) for i in range(n)]

dup_report = img_df.groupby("duplicate_group_id").filter(lambda g: len(g) > 1).sort_values("duplicate_group_id")
dup_report.to_csv(MANIFESTS / "duplicate_report.csv", index=False)
print(f"Ảnh nằm trong nhóm trùng/near-duplicate: {len(dup_report)} / {n}")
print(f"Số nhóm trùng: {dup_report['duplicate_group_id'].nunique() if len(dup_report) else 0}")

### Bước 6 — Chia train/val/test chống leakage
75/15/10 theo `duplicate_group_id`, cân bằng theo từng nguồn. `openfield_bd` bị loại khỏi pool chính, đưa toàn bộ vào `test_outdomain_openfield` (test ngoài miền — license chưa xác nhận, không train).

In [ ]:
main_df = img_df[img_df["source"] != "openfield_bd"].copy()
outdomain_df = img_df[img_df["source"] == "openfield_bd"].copy()

group_source = main_df.groupby("duplicate_group_id")["source"].agg(lambda s: s.value_counts().idxmax())

groups_by_source = defaultdict(list)
all_groups = group_source.index.tolist()
random.Random(42).shuffle(all_groups)
for g in all_groups:
    groups_by_source[group_source[g]].append(g)

split_assignment = {}
for source, glist in groups_by_source.items():
    n_g = len(glist)
    n_train = int(n_g * 0.75)
    n_val = int(n_g * 0.15)
    for g in glist[:n_train]:
        split_assignment[g] = "train"
    for g in glist[n_train:n_train + n_val]:
        split_assignment[g] = "val"
    for g in glist[n_train + n_val:]:
        split_assignment[g] = "test"

main_df["split"] = main_df["duplicate_group_id"].map(split_assignment)
outdomain_df["split"] = "test_outdomain_openfield"

final_df = pd.concat([main_df, outdomain_df], ignore_index=True)
print(final_df.groupby(["source", "split"]).size().unstack(fill_value=0))

In [ ]:
def dest_split_dir(split):
    if split == "test_outdomain_openfield":
        return PROCESSED / "test_outdomain_openfield"
    return PROCESSED / split


for _, row in final_df.iterrows():
    src_img = Path(row["path"])
    src_lbl = src_img.parent.parent / "labels" / (src_img.stem + ".txt")
    dst_dir = dest_split_dir(row["split"])
    (dst_dir / "images").mkdir(parents=True, exist_ok=True)
    (dst_dir / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_img, dst_dir / "images" / src_img.name)
    if src_lbl.exists():
        shutil.copy2(src_lbl, dst_dir / "labels" / src_lbl.name)

final_df[["source", "filename", "sha256", "phash", "duplicate_group_id", "split"]].to_csv(
    MANIFESTS / "split_manifest.csv", index=False
)
print("Đã copy xong vào", PROCESSED)

### Bước 7 — Manifest cuối: `data.yaml`, thống kê class/nguồn, license, báo cáo

In [ ]:
def count_classes(split_dir):
    counts = defaultdict(int)
    labels_dir = split_dir / "labels"
    if not labels_dir.exists():
        return counts
    for txt in labels_dir.glob("*.txt"):
        for line in txt.read_text().splitlines():
            if line.strip():
                counts[TARGET_CLASSES[int(line.split()[0])]] += 1
    return counts


rows = []
for split_name in ["train", "val", "test", "test_outdomain_openfield"]:
    split_dir = dest_split_dir(split_name)
    n_images = len(list((split_dir / "images").glob("*"))) if (split_dir / "images").exists() else 0
    rows.append({"split": split_name, "n_images": n_images, **count_classes(split_dir)})

class_dist_df = pd.DataFrame(rows).fillna(0)
class_dist_df.to_csv(MANIFESTS / "class_distribution.csv", index=False)
print(class_dist_df)

source_dist = final_df.groupby(["split", "source"]).size().unstack(fill_value=0)
source_dist.to_csv(MANIFESTS / "source_distribution.csv")
print("\n", source_dist)

In [ ]:
data_yaml = {
    "path": str(PROCESSED),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {i: n for i, n in enumerate(TARGET_CLASSES)},
}
with open(PROCESSED / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)
print((PROCESSED / "data.yaml").read_text())
print("\nLưu ý: 'test_outdomain_openfield' KHÔNG nằm trong data.yaml — chỉ dùng đánh giá thủ công ngoài miền.")

In [ ]:
src_manifest_path = RAW_INPUT.parent / "manifests" / "sources.csv"
license_lines = ["# Licenses — tomato_ripeness_v1\n"]
if src_manifest_path.exists():
    src_df = pd.read_csv(src_manifest_path)
    for _, r in src_df.iterrows():
        if r["dataset_id"] in CLASS_MAPPING:
            note = ""
            if r["dataset_id"] == "openfield_bd":
                note = " — CHỈ dùng test ngoài miền, KHÔNG train (license chưa xác nhận)"
            license_lines.append(f"- **{r['name']}** (`{r['dataset_id']}`): {r['license']}{note}. Nguồn: {r['url']}")
else:
    license_lines.append("(Không tìm thấy sources.csv gốc trong input — điền thủ công.)")

(MANIFESTS / "licenses.md").write_text("\n".join(license_lines), encoding="utf-8")
print((MANIFESTS / "licenses.md").read_text())

In [ ]:
report_lines = [
    "# tomato_ripeness_v1 — Dataset Report",
    f"Sinh tự động bởi `01_build_tomato_ripeness_v1.ipynb` — {pd.Timestamp.now().date()}",
    "",
    "## Convert theo nguồn",
    stats_df.to_string(),
    "",
    "## Phân bố ảnh theo split/nguồn",
    source_dist.to_string(),
    "",
    "## Phân bố class theo split",
    class_dist_df.to_string(index=False),
    "",
    "## Trùng lặp",
    f"- Tổng ảnh: {n}",
    f"- Ảnh nằm trong nhóm trùng/near-duplicate: {len(dup_report)}",
    f"- Số nhóm trùng: {dup_report['duplicate_group_id'].nunique() if len(dup_report) else 0}",
    "",
    "## Cảnh báo cần xử lý thủ công",
    f"- `review_required.csv`: {len(all_review_rows)} ảnh có box class REVIEW_REQUIRED (vd \"Reddish\" của AgRobTomato) bị loại khỏi label.",
    f"- `rejected_images.csv`: {len(all_rejected_rows)} ảnh bị loại hoàn toàn (không còn box hợp lệ sau remap, hoặc thiếu annotation khớp ảnh).",
    "- Class-id của `tomato_plantfactory` và `openfield_bd` là GIẢ ĐỊNH (không tìm thấy classes.txt trong dataset gốc ở lần chạy đầu) — đã cố xác nhận trực quan ở Bước 8, nhưng cần double-check nếu train ra kết quả bất thường.",
    "- Ảnh không có annotation khớp (không tìm thấy xml/txt/json tương ứng) bị loại hoàn toàn khỏi dataset, không được giữ làm negative sample.",
    "- `openfield_bd` license chưa xác nhận -> nằm ở `test_outdomain_openfield/`, KHÔNG có trong `data.yaml` train/val/test.",
]
report = "\n".join(report_lines)
(MANIFESTS / "dataset_report.md").write_text(report, encoding="utf-8")
print(report)

### Bước 8 — Xác nhận trực quan (bắt buộc trước khi tin tưởng dataset)
Vẽ bounding box lên ảnh mẫu. Đặc biệt kiểm tra kỹ 2 ảnh nhóm `tomato_plantfactory` và `openfield_bd` (test ngoài miền) vì class-id của chúng là **giả định** ở Bước 3 — nếu box tô sai màu/sai nhãn so với thực tế (vd box "fruit_ripe" nhưng quả trong ảnh còn xanh), phải sửa `ASSUMED_ID_TO_NAME` ở Bước 3 rồi chạy lại toàn bộ notebook.

In [ ]:
def show_samples(split_dir, source_filter=None, n=6, title=""):
    images_dir = split_dir / "images"
    labels_dir = split_dir / "labels"
    all_imgs = sorted(images_dir.glob("*"))
    if source_filter:
        all_imgs = [p for p in all_imgs if p.name.startswith(source_filter + "__")]
    if not all_imgs:
        print(f"[{title}] Không có ảnh để hiển thị (source_filter={source_filter}).")
        return
    sample = random.Random(42).sample(all_imgs, min(n, len(all_imgs)))

    fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
    axes = [axes] if len(sample) == 1 else axes
    colors = ["lime", "orange", "red"]
    for ax, img_path in zip(axes, sample):
        img = Image.open(img_path)
        w, h = img.size
        ax.imshow(img)
        label_path = labels_dir / (img_path.stem + ".txt")
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if not line.strip():
                    continue
                cid, xc, yc, bw, bh = line.split()
                cid = int(cid)
                xc, yc, bw, bh = map(float, (xc, yc, bw, bh))
                xmin, ymin = (xc - bw / 2) * w, (yc - bh / 2) * h
                rect = patches.Rectangle((xmin, ymin), bw * w, bh * h, linewidth=2,
                                          edgecolor=colors[cid], facecolor="none")
                ax.add_patch(rect)
                ax.text(xmin, max(ymin - 5, 0), TARGET_CLASSES[cid], color=colors[cid], fontsize=9, weight="bold")
        ax.set_title(img_path.name, fontsize=7)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    save_path = MANIFESTS / f"audit_samples_{title.replace(' ', '_')}.png"
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", save_path)


show_samples(PROCESSED / "train", title="tong_hop_train")
show_samples(PROCESSED / "train", source_filter="tomato_plantfactory", title="XAC_NHAN_tomato_plantfactory")
show_samples(PROCESSED / "test_outdomain_openfield", title="XAC_NHAN_openfield_bd")

## Kết quả
- `tomato_ripeness_v1` hoàn chỉnh nằm ở `/kaggle/working/ai/datasets/processed/tomato_ripeness_v1/` với `train/`, `val/`, `test/`, `test_outdomain_openfield/`, và `data.yaml`.
- Toàn bộ manifest (mapping, thống kê, license, báo cáo, ảnh audit) nằm ở `/kaggle/working/ai/datasets/manifests/`.

## Trước khi Save Version — checklist
- [ ] Bước 2: tên class gốc in ra khớp với `name_map` ở Bước 3 (không có class lạ bị âm thầm drop).
- [ ] Bước 8: ảnh mẫu `tomato_plantfactory` và `openfield_bd` gắn đúng nhãn màu quả thật (xác nhận `ASSUMED_ID_TO_NAME`).
- [ ] `dataset_report.md`: số ảnh/box mỗi split hợp lý, không có nguồn nào `images_kept = 0`.
- [ ] `review_required.csv` và `rejected_images.csv`: xem qua để biết có bao nhiêu ảnh bị loại và vì sao.

Sau khi hài lòng, bấm **Save Version → Save & Run All (Commit)** để `tomato_ripeness_v1` trở thành Kaggle Dataset độc lập.

## Bước tiếp theo
`02_train_ripeness_baseline.ipynb` (chưa tạo): Add Input dataset vừa tạo, train YOLO nano (imgsz 640, epochs ~100, patience 20, seed 42) trên `train/val`, đánh giá riêng trên `test` và `test_outdomain_openfield` để đo domain gap, theo đúng cấu hình baseline trong `TONG_HOP_YEU_CAU_VA_DE_XUAT_AI_SMART_GREENHOUSE.docx`.